In [1]:
import pickle
import re
import pysam
import numpy as np
import pandas as pd

from typing import Any, Dict, List, Tuple, Union
from collections import defaultdict, Counter

In [2]:
class CallMethVector:
    """
     return vector of methylated sites
    """

    def __init__(self, bam_file: str = None, genome_db: str = None, 
                 contig: str = None, start=None, end=None, cg_only: bool = False, 
                 min_base_quality: int = 20, min_mapping_quality: int = 30, 
                 return_queue=None, filter_duplicates=True, fully_contain=True):
        
        self.bam_file  = bam_file
        self.input_bam = pysam.AlignmentFile(self.bam_file, 'rb', require_index=True)
        self.genome_db = f'{genome_db}/'
        self.mate_flag_dict = {67: 131, 323: 387, 115: 179, 371: 435, 
                               131: 67, 387: 323, 179: 115, 435: 371}
        self.meth_state_dict= {('C','C'): [True, 1], ('C','T'): [True, 0], 
                               ('G','G'): [True, 1], ('G','A'): [True, 0]}
        
        self.contig = contig
        self.start  = start
        self.end    = end
        self.cg_only= cg_only
        self.fully_contain = fully_contain
        
        self.filter_duplicates  = filter_duplicates
        self.min_base_quality   = min_base_quality
        self.min_mapping_quality= min_mapping_quality
        
        self.chunk_size = 10000
        self.return_queue   = return_queue


    def call_meth(self):
        try:
            chrom_seq = self.get_ref_sequence(f'{self.genome_db}{self.contig}.pkl')
        except FileNotFoundError:
            self.return_queue.put([])
            print(f'{self.contig} not found in BSBolt DB, Methylation Calls for {self.contig} skipped. Methylation '
                  f'values should be called using the same DB used for alignment.')
            self.return_queue.put([])
        else:
            self.call_contig(chrom_seq)


    def align_filter(self, align):
        is_mapped  = (not align.is_unmapped)
        is_primary  = not (align.is_secondary or align.is_supplementary)
        is_qualified = (not align.is_qcfail) and align.mapping_quality >= self.min_mapping_quality
        not_duplicate = not (align.is_duplicate and self.filter_duplicates)
        is_fully_align = all(op in (0, 7, 8) for op, length in align.cigar)
        return is_mapped and is_primary and is_qualified and is_fully_align and not_duplicate


    def call_contig(self, chrom_seq: str):
        """Iterates through bam reads call methylation along vectors. When a read overlaps a site where methylation is
         called the site with a higher quality is taken. If overlapping sites with the same quality are observed
         the first observed site is reported. If an overlapping site is reported as a mismatch only the site with
         a methylation call is reported (this should be extremely rare but is observed in the test cases)
        """
        # set search pattern for only CG sites or all Cs
        pattern_list = ('CG', 'CG') if self.cg_only else ('C', 'G')
        # iterate through pileup
        contig_chunk = []
        meth_vectors = {}
        
        for aligned_read in self.input_bam.fetch(contig=self.contig, start=self.start, end=self.end, multiple_iterators=True):
            if not self.align_filter(aligned_read):
                continue
            if self.fully_contain and (aligned_read.reference_start < self.start or aligned_read.reference_end > self.end):
                continue # exclude the partially overlapped reads
            
            # get sequence around pileup site
            ref_seq = chrom_seq[aligned_read.reference_start - 1: aligned_read.reference_end + 2].upper()
            if aligned_read.get_tag('YS') == 'W_C2T': # only for directional library
                search_pattern, ref_base, strand = pattern_list[0], 'C', 'Watson'
                offset = 0
            else:
                search_pattern, ref_base, strand = pattern_list[1], 'G', 'Crick'
                offset = 1 if self.cg_only else 0
            
            search_res = [match.start() + offset + aligned_read.reference_start - 1 for match in 
                          re.finditer(search_pattern, ref_seq)]
            
            # if reference sequence not found proceed to next sequence
            if not search_res:
                continue
            
            meth_calls = self.call_vector(aligned_read, set(search_res), ref_base)
            
            if not meth_calls[0]:
                continue
            
            processed_vector = self.process_meth_vector(aligned_read, meth_calls, strand, meth_vectors)
            
            if processed_vector:
                contig_chunk.append(processed_vector)
            
            if len(contig_chunk) == self.chunk_size:
                self.return_queue.put((self.contig, contig_chunk))
                contig_chunk = []

        # process reads that didn't have a pair with a observed methylation site
        for call in meth_vectors.values():
            contig_chunk.append((call['read_name'], call['calls'][1][0], call['calls'][1][-1],
                                 np.asarray(call['calls'][0]), np.asarray(call['calls'][1]), call['flag'],
                                 self.mate_flag_dict[call['flag']], call['strand']))
        if contig_chunk:
            self.return_queue.put((self.contig, contig_chunk))


    def call_vector(self, aligned_read: pysam.AlignedRead, position_set: set, ref_base: str) -> List:
        meth_calls   = [[], [], []]
        reference_consumers = {0, 2, 3, 7, 8}
        query_consumers     = {0, 1, 4, 7, 8}
        # set relative to genomic position so add reference start and one since capturing the first base
        ref_pos     = aligned_read.reference_start
        query_seq   = aligned_read.query_sequence
        query_qual  = aligned_read.query_qualities
        query_pos   = 0
        
        for cigar_type, cigar_count in aligned_read.cigartuples:
            if cigar_type in reference_consumers and cigar_type in query_consumers:
                for _ in range(cigar_count):
                    if ref_pos in position_set:
                        pos_qual = query_qual[query_pos]
                        if pos_qual > self.min_base_quality:
                            query_base = query_seq[query_pos]
                            call_made, meth_state = self.get_meth_call(ref_base, query_base)
                            if call_made:
                                meth_calls[0].append(meth_state)
                                meth_calls[1].append(ref_pos)
                                meth_calls[2].append(pos_qual)
                    ref_pos += 1
                    query_pos += 1
            elif cigar_type in reference_consumers and cigar_type not in query_consumers:
                ref_pos += cigar_count
            elif cigar_type in query_consumers and cigar_type not in reference_consumers:
                query_pos += cigar_count
        return meth_calls

    def process_meth_vector(self, aligned_read: pysam.AlignedRead, meth_calls: List,
                            strand: str, meth_vectors: Dict[str, Any]) -> Union[None, Tuple]:
        if aligned_read.is_proper_pair:
            mate_flag = self.mate_flag_dict[aligned_read.flag]
            mate_pair_label = f'{aligned_read.query_name}_{mate_flag}_{aligned_read.next_reference_start}'
            if mate_pair_label in meth_vectors:
                paired_calls = meth_vectors.pop(mate_pair_label)['calls']
                paired_calls[0].extend(meth_calls[0])
                paired_calls[1].extend(meth_calls[1])
                paired_calls[2].extend(meth_calls[2])
                paired_calls = self.clean_overlap(paired_calls)
                return (aligned_read.query_name, paired_calls[1][0], paired_calls[1][-1],
                        np.array(paired_calls[0]), np.array(paired_calls[1]),
                        aligned_read.flag, mate_flag, strand)
            else:
                vector_label = f'{aligned_read.query_name}_{aligned_read.flag}_{aligned_read.reference_start}'
                meth_vectors[vector_label] = {'calls': meth_calls,
                                                     'read_name': aligned_read.query_name,
                                                     'flag': aligned_read.flag, 'strand': strand}
                return None
        else:
            return (aligned_read.query_name, meth_calls[1][0], meth_calls[1][-1],
                    np.array(meth_calls[0]), np.array(meth_calls[1]), aligned_read.flag, None, strand)


    def get_meth_call(self, ref_base: str, base_call: str) -> Tuple[bool, Union[None, int]]:
        """
        Methylation for a C relative to the sense strand of the reference can only be called using watson reads,
        and G with crick reads
        Arguments
            nucleotide (str): reference nucleotide
            base_call (collections.Counter): watson nucleotides are Uppercase and crick nucleotides lowercase
        Returns:
            methylation call dictionary
        """
        # call cytosine with watson
        try:
            tmp_call = self.meth_state_dict[(ref_base, base_call)]
        except:
            return False, None
        else:
            return tmp_call


    @staticmethod
    def clean_overlap(meth_calls: List) -> List:
        cleaned_calls = {}
        for meth_call, pos, qual in zip(meth_calls[0], meth_calls[1], meth_calls[2]):
            if pos not in cleaned_calls:
                cleaned_calls[pos] = (meth_call, qual)
            else:
                if qual > cleaned_calls[pos][1]:
                    cleaned_calls[pos] = (meth_call, qual)
        return [[call[0] for call in cleaned_calls.values()], list(cleaned_calls.keys())]


    @staticmethod
    def get_ref_sequence(path: str) -> str:
        """load serialized reference file from path
        """
        with open(path, 'rb') as genome_file:
            return pickle.load(genome_file)


In [3]:
import queue
wgbs_file  = "/home/wbguo/iproject/BSReadSim/test/data/WGBS/ERR2359938.mkdup.sorted.bam"
genome_db  = "/home/wbguo/iproject/BSReadSim/test/data/idx/GRCh38/"
region_bed = "/home/wbguo/iproject/BSReadSim/test/data/WGBS/ERR2359938.chr21.d20.l500.merged.bed"

In [4]:
region_df = pd.read_csv(region_bed, header=None, sep = '\t')

In [5]:
region_df

,0,1,2
0,chr21,5220063,5220884
1,chr21,5227568,5228196
2,chr21,5228197,5229065
3,chr21,5232044,5232815
4,chr21,5236579,5237877
...,...,...,...
4580,chr21,46669371,46669971
4581,chr21,46670010,46670650
4582,chr21,46670803,46671314
4583,chr21,46674370,46674995


In [ ]:
feature_chunk = []
target_chunk  = []

chunk_size = 10**4
i = 0

for index, row in region_df.iterrows():
    contig_id, start, end = row
    # construct the context dict
    context_dict = get_region_context(contig_id, start, end)
    
    return_queue = queue.Queue()
    vector_caller= CallMethVector(bam_file = wgbs_file, genome_db= genome_db, 
                                  contig = contig_id, start=start, end=end, return_queue = return_queue)
    
    vector_caller.call_meth()
    while True:
        contig_id, data = return_queue.get()
        # process data
        feture_list, target_list = prepare_data(data, pos_dict, window_size)
        feature_chunk.extend(feature_list)
        target_chunk.extend(target_list)
        
        if return_queue.empty():
            break
    
    if len(read_chunk) > chunk_size:
        save_chunk(feature_chunk, target_chunk, i)
        read_chunk = []
        i += 1

In [ ]:
def get_cg_context(base, base_d1, base_d2):
    '''input the base and surrounding, output context'''
    base_context_table = {'C': np.array([[5,3], [1,1]]),  'G': np.array([[-5,-3],[-1,-1]])}
    
    if base == "C":
        flag_d1 = int(base_d1 == "G")
        flag_d2 = int(base_d2 == "G")
    elif base == "G":
        flag_d1 = int(base_d1 == "C")
        flag_d2 = int(base_d2 == "C")
    else:
        return None
    return int(base_context_table[base][flag_d1, flag_d2])

def get_region_context(contig_seq, start, end):
    pos_context = {}
    for i in range(start, end+1):
        base = contig_seq[i:(i+1)]
        if base not in {'C', 'G'}:
            continue
        updown  = 1 if base == "C" else -1
        base_d1 = contig_seq[i + 1*updown]
        base_d2 = contig_seq[i + 2*updown]
        pos_context[i] = get_cg_context(base, base_d1, base_d2)
    
    return pos_context

In [ ]:
def prepare_data(data, pos_context, window_size):
    feature_list = []
    target_list  = []
    
    for rec in data:
        target = rec[3]
        strand = rec[7]
        pos_arr= rec[4]
        
        # construct meth
        meth   = [meth_dict[pos] for pos in pos_arr]
        # actually can consider watson and crick separately
        dist = np.concatenate([np.array([0]), np.diff(pos_arr, axis = 0)])
        context   = np.zeros(pos.size)
        embedding = np.zeros((768, pos.size))
        for ix, i in enumerate(pos):
            context[ix] = pos_context[i]
            local_seq = contig_seq[(i-window_size): (i+1+window_size)]
            embedding[:,ix] = embedding_dict[local_seq]
        
        feature = np.concatenate([meth, dist, context, embedding], axis=0)
        feature_list.append(feature)
        target_list.append(target)
    
    return feature_list, target_list

In [ ]:
index = 0
row = region_df.iloc[index,:]

In [ ]:
contig_id, start, end = row
return_queue = queue.Queue()
vector_caller= CallMethVector(bam_file = wgbs_file, genome_db= genome_db, 
                              contig = contig_id, start=start, end=end, return_queue = return_queue)

In [ ]:
vector_caller.call_meth()

In [ ]:
while True:
    contig, data = return_queue.get()
    df = 

In [ ]:
contig

In [ ]:
data[0]

In [ ]:
df = pd.DataFrame(data)
df

In [ ]:
df.value_counts(subset=[7])

In [ ]:
def get_feature_target(data):
    for item in data:
        

# checked true

In [ ]:
import queue
#input_file = "/u/home/h/hongxian/project-pellegrini/BS_genotype/data/WGBS/ERR2359938.sorted.bam"
#genome_database = "/u/home/h/hongxian/project-pellegrini/BS_genotype/data/WGBS/"
#input_file = "/u/home/h/hongxian/iproject/Wenbin/ERR2359938_filtered.bam"
#genome_database = "/u/home/h/hongxian/project-pellegrini/BS_genotype/PGP-UK/idx/bsbolt/"
input_file = "/home/wbguo/iproject/BSReadSim/test/data/sim/pe_d/sim.mkdup.sorted.bam"
genome_database = "/home/wbguo/iproject/BSReadSim/test/data/idx/BSB_test/"
contig = 'chr10'
return_queue = queue.Queue()
vector_caller = CallMethylationVector(input_file = input_file, genome_database = genome_database, contig = contig, return_queue = return_queue)

In [ ]:
# find the depth region, consecutive depth with >= 20


In [ ]:
vector_caller.call_methylation()

In [ ]:
all_vectors = []
while True:
    values = return_queue.get()
    all_vectors.extend(values[1])
    if return_queue.empty():
        break

In [ ]:
all_vectors

In [ ]:
for vec in all_vectors:
#     if vec[0] == '797f_chr10':
#         break
    if vec[-1] == 'crick' and 'chr10' in vec[0]:
        break

In [ ]:
vec

In [ ]:
vec[4][np.where(vec[3])]

In [ ]:
def align_filter(align, mapq_thre =20):
    is_mapped  = (not align.is_unmapped)
    # the primary and supplementary is removed in read_pair_generator
    #is_primary = not (align.is_secondary or align.is_supplementary)
    is_qualified = (not align.is_qcfail) and align.mapq >= mapq_thre
    not_duplicate = not align.is_duplicate
    is_fully_align = all(op in (0, 7, 8) for op, length in align.cigar)
    return is_mapped and is_qualified and is_fully_align and not_duplicate

def read_pair_generator(bam_file, region_string=None):
    """
    Generate read pairs in a BAM file or within a region string.
    Reads are added to read_dict until a pair is found.
    """
    read_dict = defaultdict(lambda: [None, None])
    bam = pysam.AlignmentFile(bam_file, "rb")
    #bam.reset()
    
    for align in bam.fetch(until_eof=True, region=region_string):        
        if not align.is_proper_pair or align.is_secondary or align.is_supplementary:
            continue
        
        qname = align.query_name
        if qname not in read_dict.keys():
            read_dict[qname][int(align.is_read2)] = align
        else:
            read_dict[qname][int(align.is_read2)] = align
            yield read_dict[qname]
            del read_dict[qname]

In [ ]:
for read1, read2 in read_pair_generator(input_file, "chr10"):
    print(read1.is_reverse == read2.is_reverse)

In [ ]:
for read1, read2 in read_pair_generator(wgbs_file, "chr10"):
    if read1.is_reverse != read2.is_reverse:
        print('well')